In [ ]:
from essential.gpu_utils import select_best_gpus

select_best_gpus()

import scanpy as sc
from sklearn.cluster import KMeans
import plotnine as gg
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE as TSNE_sklearn
from sklearn.cluster import KMeans
from tqdm import tqdm
import itertools
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import seaborn as sns
from sklearn.metrics import precision_recall_curve

# from sklearn.manifold import TSNE
from cuml.manifold import TSNE
import numpy as np
import pandas as pd
import seaborn as sns
import jax.numpy as jnp
import plotly.express as px
import matplotlib.pyplot as plt
import scipy.stats as stats

from essential.stats import MMDTestJax, MMDTest
from essential.data import load_fitness_data
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
import matplotlib.colors as mcolors
import json

pd.set_option("display.max_columns", 500)

SHARED_THEME = gg.theme(
    axis_text=gg.element_text(size=6),
    axis_title=gg.element_text(size=7),
    figure_size=(3, 2),
    title=gg.element_text(size=7),
    legend_text=gg.element_text(size=6),
)


def compute_pairwise(df1, df2=None, metric="cosine"):
    if metric == "cosine":
        metric_func = cosine_similarity
    elif metric == "euclidean":
        metric_func = euclidean_distances
    else:
        raise ValueError(f"Metric {metric} not supported")

    if df2 is None:
        pairwise_d = metric_func(df1)
        pairwise_d = pd.DataFrame(pairwise_d, index=df1.index, columns=df1.index)
    else:
        pairwise_d = metric_func(df1, df2)
        pairwise_d = pd.DataFrame(pairwise_d, index=df1.index, columns=df2.index)
    return pairwise_d


def plot_similarity_matrix(
    df, row_metadata, row_color_column, cmap="viridis", similarity_label="cosine similarity"
):
    """
    Plots a seaborn clustermap with a row colorbar.

    df: pd.DataFrame representing the similarity matrix.
    row_metadata: pd.DataFrame containing metadata for rows (must match df's rows).
    row_color_column: str, the column in row_metadata to map to colors.
    """
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    import matplotlib.cm as cm
    import seaborn as sns

    cmap_obj = plt.get_cmap(cmap)
    norm = mcolors.Normalize(
        vmin=row_metadata[row_color_column].min(), vmax=row_metadata[row_color_column].max()
    )
    row_colors = row_metadata[row_color_column].map(lambda x: mcolors.to_hex(cmap_obj(norm(x))))

    g = sns.clustermap(
        df,
        row_colors=row_colors,
        xticklabels=False,
        yticklabels=False,
        cbar_pos=(0.02, 0.8, 0.05, 0.18),
        cbar_kws={"label": similarity_label},
    )

    cbar_ax = g.fig.add_axes([0.02, 0.55, 0.05, 0.18])
    cb = plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap_obj), cax=cbar_ax)
    cb.set_label(row_color_column)

    return g

# Imports; preprocessing

In [ ]:
with open("gene_llm_annotations.json", "r") as file:
    automated_gene_annotations = json.load(file)

automated_gene_annotations = pd.Series(automated_gene_annotations).to_frame("automated_annotation")

In [ ]:
fitness_df = load_fitness_data()
fitness_df_gene = fitness_df.groupby("gene")[["T1", "T2", "T3", "T4"]].mean()
fitness_df_ctrl = fitness_df_gene.loc[
    lambda x: x.index.fillna("NA").str.startswith("Control")
].mean(0)
fitness_df_scores = (fitness_df_gene - fitness_df_ctrl) / fitness_df_ctrl
fitness_df_scores.columns = ["T1_score", "T2_score", "T3_score", "T4_score"]
fitness_df_gene = fitness_df_gene.merge(fitness_df_scores, left_index=True, right_index=True)

flux_df = pd.read_csv("/workspace/experiments/01232026_fba/data/moma_fluxes.csv", index_col=0)
worker_df = pd.read_csv("/workspace/experiments/01232026_fba/data/worker_ids.csv", index_col=0)
wt_flux = pd.read_csv("/workspace/experiments/01232026_fba/data/wt_fluxes_0.csv", index_col=0)
growth_df = (
    pd.read_csv("/workspace/experiments/01232026_fba/data/fba_growth_ratios.csv", index_col=0)
    .merge(fitness_df_gene, left_index=True, right_index=True)
    .assign(
        fba_growth_type=lambda x: pd.Categorical(
            np.where(x["growth_ratio"] >= 0.5, "high", "low"), categories=["low", "high"]
        ),
        growth_score=lambda x: (x["growth"] - x["growth_wt"]) / x["growth_wt"],
        is_predicted_essential=lambda x: x["growth_ratio"] < 0.5,
        is_experimental_essential=lambda x: x["T4"] < -3.0,
    )
    .merge(worker_df, left_index=True, right_index=True)
)

# preprocessing + low-dimensional embedding
reaction_std = flux_df.std(axis=0)
flux_df_ = flux_df.loc[:, reaction_std >= 1e-3]
print(flux_df_.shape)
wt_flux_df_ = wt_flux.loc[flux_df_.columns.values]

flux_df_upperb = np.quantile(flux_df_, 0.95, axis=0)
flux_df_lowerb = flux_df_.min(axis=0)
flux_df_ = np.clip(flux_df_, flux_df_lowerb, flux_df_upperb, axis=1)
flux_df_ = (flux_df_ - flux_df_lowerb) / (flux_df_upperb - flux_df_lowerb + 1e-6)
# flux_df_bin = (flux_df_ > 0.5).astype(float)

wt_flux_df_ = np.clip(wt_flux_df_.T, flux_df_lowerb, flux_df_upperb, axis=1)
wt_flux_df_ = (wt_flux_df_ - flux_df_lowerb) / (flux_df_upperb - flux_df_lowerb + 1e-6)
wt_flux_df_ = wt_flux_df_.T
# wt_flux_df_bin = (wt_flux_df_ > 0.5).astype(float)

pca_ = PCA(n_components=25)
flux_df_pca = pca_.fit_transform(flux_df_)
print(pca_.explained_variance_ratio_.sum())

tsne_ = TSNE(n_components=2, metric="l1", init="pca")
flux_df_tsne = tsne_.fit_transform(flux_df_)
flux_df_tsne_vals = flux_df_tsne.values

flux_df_reduced = (
    pd.DataFrame(flux_df_pca[:, :2], index=flux_df_.index, columns=["PC1", "PC2"])
    .reset_index()
    .rename(columns={"index": "gene_name"})
)
flux_df_reduced["t-SNE1"] = flux_df_tsne_vals[:, 0]
flux_df_reduced["t-SNE2"] = flux_df_tsne_vals[:, 1]
flux_df_reduced = flux_df_reduced.merge(
    growth_df, left_on="gene_name", right_index=True, how="left"
)

# clustering
gene_clusters = KMeans(n_clusters=10).fit_predict(flux_df_)
flux_df_reduced["metabolic_cluster"] = gene_clusters
flux_df_reduced["metabolic_cluster"] = flux_df_reduced["metabolic_cluster"].astype(str)

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata_case = sc.read_h5ad("/workspace/data/251117_genomescale_CRISPRi/adata_case.annotated.h5ad")

adata.obs = adata.obs.merge(flux_df_reduced, left_on="gene", right_on="gene_name", how="left")
adata_case.obs = adata_case.obs.merge(
    flux_df_reduced, left_on="gene", right_on="gene_name", how="left"
)
adata.obs["UMAP1"] = adata.obsm["X_umap"][:, 0]
adata.obs["UMAP2"] = adata.obsm["X_umap"][:, 1]

adata_case.obs["UMAP1"] = adata_case.obsm["X_umap"][:, 0]
adata_case.obs["UMAP2"] = adata_case.obsm["X_umap"][:, 1]

transcript_df = []
transcript_case_df = []
z_transcript_df = []
z_transcript_case_df = []
gene_names = []
gene_case_names = []
selected_genes = np.intersect1d(flux_df_.index.astype(str).values, adata.var_names)
adata_ = adata[:, selected_genes]
adata_case_ = adata_case[:, selected_genes]

for gene in tqdm(adata_.obs["gene"].unique()):
    adata_gene = adata_[adata_.obs["gene"] == gene]
    X_gene = adata_gene.layers["cp10k"].toarray()
    if X_gene.shape[0] > 0:
        gene_names.append(gene)
        transcript_df.append(X_gene.mean(axis=0))
        z_transcript_df.append(adata_gene.obsm["X_scVI"].mean(axis=0))

    adata_gene_case = adata_case_[adata_case_.obs["gene"] == gene]
    X_gene_case = adata_gene_case.layers["cp10k"].toarray()
    if X_gene_case.shape[0] > 0:
        gene_case_names.append(gene)
        transcript_case_df.append(X_gene_case.mean(axis=0))
        z_transcript_case_df.append(adata_gene_case.obsm["X_scVI"].mean(axis=0))
transcript_df = pd.DataFrame(transcript_df, index=gene_names)
transcript_case_df = pd.DataFrame(transcript_case_df, index=gene_case_names)
z_transcript_df = pd.DataFrame(z_transcript_df, index=gene_names)
z_transcript_case_df = pd.DataFrame(z_transcript_case_df, index=gene_case_names)
# transcript_df = transcript_df / (1e-6 + transcript_df.max(axis=0))
transcript_df_ctrl = transcript_df.loc[lambda x: x.index.str.startswith("Control")]
transcript_case_df_ctrl = transcript_case_df.loc[lambda x: x.index.str.startswith("Control")]


transcript_d_to_all_ctrl = compute_pairwise(transcript_df, transcript_df_ctrl)
transcript_d_to_ctrl = transcript_d_to_all_ctrl.mean(1).to_frame("transcript_d_to_ctrl")

In [ ]:
# transcriptomic-derived annotations
gene_transcript_annotation = (
    adata_case.obs.groupby("gene")["annotated_leiden_case"]
    .agg(lambda x: x.mode()[0])
    .to_frame("transcript_annotation")
)
gene_transcript_annotation_coarse = (
    adata_case.obs.groupby("gene")["annotated_leiden_case_coarse"]
    .agg(lambda x: x.mode()[0])
    .to_frame("transcript_annotation_coarse")
)
gene_transcript_annotation = gene_transcript_annotation.merge(
    gene_transcript_annotation_coarse, left_index=True, right_index=True, how="left"
)

### Design proper strategy to embed transcriptomics data

In [ ]:
transcript_case_df_pca = PCA(n_components=50).fit_transform(transcript_case_df)
transcript_case_df_tsne = TSNE_sklearn(n_components=2, metric="euclidean").fit_transform(
    transcript_case_df
)
transcript_case_df_pca_ = pd.DataFrame(transcript_case_df_pca, index=transcript_case_df.index)
transcript_pairwise_case = compute_pairwise(transcript_case_df_pca_, metric="cosine")

transcript_case_df_vis = pd.DataFrame(
    {
        "t-SNE1": transcript_case_df_tsne[:, 0],
        "t-SNE2": transcript_case_df_tsne[:, 1],
    },
    index=transcript_case_df.index,
).merge(gene_transcript_annotation, left_index=True, right_index=True, how="left")

(
    gg.ggplot(
        transcript_case_df_vis, gg.aes(x="t-SNE1", y="t-SNE2", color="transcript_annotation_coarse")
    )
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(legend_key_size=8, figure_size=(4, 2))
)

In [ ]:
z_transcript_tsne = TSNE_sklearn(n_components=2).fit_transform(z_transcript_case_df.values)
z_transcript_tsne_vis = pd.DataFrame(
    z_transcript_tsne, index=z_transcript_case_df.index, columns=["t-SNE1", "t-SNE2"]
)
# z_transcript_tsne = TSNE_sklearn(n_components=2).fit_transform(z_transcript_df.values)
# z_transcript_tsne_vis = pd.DataFrame(
#     z_transcript_tsne, index=z_transcript_df.index, columns=["t-SNE1", "t-SNE2"]
# )
z_transcript_tsne_vis = z_transcript_tsne_vis.merge(
    gene_transcript_annotation, left_index=True, right_index=True, how="left"
)

In [ ]:
(
    gg.ggplot(
        z_transcript_tsne_vis, gg.aes(x="t-SNE1", y="t-SNE2", color="transcript_annotation_coarse")
    )
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(legend_key_size=8, figure_size=(4, 2))
)

In [ ]:
predicted_essential_perts = growth_df.loc[growth_df["is_predicted_essential"]].index
predicted_essential_perts_inter = np.intersect1d(
    predicted_essential_perts, transcript_pairwise_case.index
)

In [ ]:
sns.clustermap(
    transcript_pairwise_case.loc[predicted_essential_perts_inter].loc[
        :, predicted_essential_perts_inter
    ],
    xticklabels=False,
    yticklabels=False,
)
plt.show()

In [ ]:
sns.clustermap(transcript_pairwise_case, xticklabels=False, yticklabels=False)

In [ ]:
# # t-SNE Plot
# fig_tsne = px.scatter(
#     flux_df_reduced,
#     x="PC1",
#     y="PC2",
#     color="growth_ratio",
#     hover_data=["gene_name"],
#     title="t-SNE of Flux Distribution",
#     template="plotly_white",
#     height=500,
#     width=500,
# )
# fig_tsne.write_html("explore_results_tsne.html")
# fig_tsne.show()

<!-- ## Idea 1: pairwise distance comparison -->

# overall properties

### fitness properties

In [ ]:
fig = (
    gg.ggplot(growth_df, gg.aes(x="growth_ratio"))
    + gg.geom_histogram(bins=100)
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.labs(x="FBA-predicted growth", y="# of genes")
)
# fig.save("growth_ratio_histogram.png", dpi=300)
fig

In [ ]:
y_gt = growth_df["is_experimental_essential"]
y_pred = growth_df["is_predicted_essential"]
tn, fp, fn, tp = confusion_matrix(y_gt, y_pred).ravel()
cm = confusion_matrix(y_gt, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Is predicted essential")
plt.ylabel("Is experimental essential")
plt.show()

print(classification_report(y_gt, y_pred))

In [ ]:
y_gt = growth_df["is_experimental_essential"]
y_score = -growth_df["growth_ratio"]

precision, recall, thresholds = precision_recall_curve(y_gt, y_score)
plot_df = pd.DataFrame({"precision": precision, "recall": recall})
fig = (
    gg.ggplot(plot_df, gg.aes(x="recall", y="precision"))
    + gg.geom_line()
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.labs(title="essential gene prediction (positives) based on FBA")
)
# fig.save("fba_precision_recall_curve.png", dpi=300)
fig

# pairwise distance comparison

### Using pertubational transcriptomics

In [ ]:
# transcript_pairwise = compute_pairwise(transcript_df)
# transcript_pairwise_case = compute_pairwise(transcript_case_df)

transcript_pairwise = compute_pairwise(z_transcript_case_df, metric="euclidean")
transcript_pairwise_case = compute_pairwise(z_transcript_case_df, metric="euclidean")

In [ ]:
sns.clustermap(
    flux_pairwise_pred_essential,
    xticklabels=False,
    yticklabels=True,
    figsize=(20, 20),
)
plt.show()

### Using FBA/MOMA

In [ ]:
flux_pairwise = compute_pairwise(flux_df_)
flux_d_to_ctrl = compute_pairwise(flux_df_, wt_flux_df_.T)
# flux_pairwise = compute_pairwise(flux_df_bin)
# flux_d_to_ctrl = compute_pairwise(flux_df_bin, wt_flux_df_bin.T)
flux_d_to_ctrl.columns = ["flux_d_to_ctrl"]

In [ ]:
growth_df["growth ratio (FBA)"] = growth_df["growth_ratio"]

In [ ]:
g = plot_similarity_matrix(
    flux_pairwise,
    growth_df,
    row_color_column="growth ratio (FBA)",
    similarity_label="flux cosine similarity",
)
# g.savefig("./flux_clustermap.png", dpi=500, bbox_inches="tight")
plt.show()

In [ ]:
predicted_essential_perts = growth_df.loc[growth_df["is_predicted_essential"]].index
flux_pairwise_pred_essential = flux_pairwise.loc[predicted_essential_perts].loc[
    :, predicted_essential_perts
]
growth_df_pred_essential = growth_df.loc[predicted_essential_perts].merge(
    flux_d_to_ctrl, left_index=True, right_index=True, how="left"
)

g = plot_similarity_matrix(
    flux_pairwise_pred_essential,
    growth_df_pred_essential,
    row_color_column="growth ratio (FBA)",
    similarity_label="flux cosine similarity",
)
# g.savefig("./flux_clustermap_essential.png", dpi=500, bbox_inches="tight")
plt.show()

# g = plot_similarity_matrix(
#     flux_pairwise_pred_essential,
#     growth_df_pred_essential,
#     row_color_column="flux_d_to_ctrl",
#     similarity_label="flux distance to control",
# )
# # g.savefig("./flux_clustermap_essential.png", dpi=500, bbox_inches="tight")
# plt.show()

#### Annotation of predicted essential genes

In [ ]:
", ".join(predicted_essential_perts.sort_values())

In [ ]:
flux_df_pred_essential = flux_df_.loc[predicted_essential_perts]
# tsne_ = TSNE(n_components=2, metric="cosine")
# flux_df_tsne = tsne_.fit_transform(flux_df_pred_essential)
# flux_df_tsne_vals = flux_df_tsne.values

# tsne_ = TSNE_sklearn(n_components=2, metric="cosine")
tsne_ = TSNE_sklearn(n_components=2)
flux_df_tsne = tsne_.fit_transform(flux_df_pred_essential)
flux_df_tsne_vals = flux_df_tsne + np.random.normal(0, 0.001, flux_df_tsne.shape)

plot_df = (
    pd.DataFrame(flux_df_tsne_vals, index=predicted_essential_perts, columns=["t-SNE1", "t-SNE2"])
    .merge(growth_df, left_index=True, right_index=True, how="left")
    .merge(gene_transcript_annotation, left_index=True, right_index=True, how="left")
    .merge(automated_gene_annotations, left_index=True, right_index=True, how="left")
)

fig1 = (
    gg.ggplot(plot_df, gg.aes(x="t-SNE1", y="t-SNE2", color="growth ratio (FBA)"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(legend_key_size=8)
)
display(fig1)

fig2 = (
    gg.ggplot(plot_df, gg.aes(x="t-SNE1", y="t-SNE2", color="automated_annotation"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(legend_key_size=8, figure_size=(4.5, 2))
)
# fig2.save("tsne_fba_automated_annotation.png", dpi=300)
display(fig2)

fig3 = (
    gg.ggplot(plot_df, gg.aes(x="t-SNE1", y="t-SNE2", color="transcript_annotation_coarse"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(legend_key_size=8, figure_size=(4, 2))
)
display(fig3)

In [ ]:
plot_df["gene_name"] = plot_df.index
fig = px.scatter(
    plot_df,
    x="t-SNE1",
    y="t-SNE2",
    color="automated_annotation",
    hover_name="gene_name",
    template="plotly_white",
    width=600,
    height=400,
)
fig.update_traces(marker=dict(size=3))
fig.show()

#### Case study: peptoglycan biosynthesis

In [ ]:
gene_group1 = ["murC", "murI", "murD", "murE"]
gene_group2 = ["murA", "murB", "murF", "murJ", "murG", "mraY"]
all_genes = np.concatenate([gene_group1, gene_group2])

sns.clustermap(
    flux_pairwise_pred_essential.loc[all_genes, all_genes],
    xticklabels=False,
    yticklabels=True,
)
plt.show()

In [ ]:
sns.clustermap(
    flux_pairwise_pred_essential,
    xticklabels=False,
    yticklabels=True,
    figsize=(20, 20),
    # vmin=0.97,
    # linewidths=0.01,
    # linecolor="gray",
)
plt.show()

#### nonessential genes

In [ ]:
predicted_nonessential_perts = growth_df.loc[~growth_df["is_predicted_essential"]].index
flux_pairwise_pred_nonessential = flux_pairwise.loc[predicted_nonessential_perts].loc[
    :, predicted_nonessential_perts
]
growth_df_pred_nonessential = growth_df.loc[predicted_nonessential_perts]

g = plot_similarity_matrix(
    flux_pairwise_pred_nonessential,
    growth_df_pred_nonessential,
    row_color_column="growth ratio (FBA)",
    similarity_label="flux cosine similarity",
)
# g.savefig("./flux_clustermap_nonessential.png", dpi=500, bbox_inches="tight")
plt.show()

In [ ]:
predicted_essential_perts = growth_df.loc[growth_df["is_predicted_essential"]].index
predicted_essential_perts_inter = np.intersect1d(
    predicted_essential_perts, transcript_pairwise.index
)
transcript_pairwise_pred_essential = transcript_pairwise.loc[predicted_essential_perts_inter].loc[
    :, predicted_essential_perts_inter
]

sns.clustermap(
    transcript_pairwise_pred_essential,
    xticklabels=False,
    yticklabels=True,
    figsize=(20, 20),
)
plt.show()

# metabolic/transcriptomic comparison

### flux preprocessing

In [ ]:
growth_df[growth_df["is_experimental_essential"]].index

In [ ]:
def construct_vis(df, n_clusters=6):
    tsne = TSNE_sklearn(n_components=2).fit_transform(df)
    df_visualization = pd.DataFrame(tsne, index=df.index, columns=["t-SNE1", "t-SNE2"])
    clusters = KMeans(n_clusters=n_clusters).fit_predict(df)
    df_visualization["cluster"] = clusters
    return df_visualization

In [ ]:
# Flux perspective
# flux_pairwise_ = compute_pairwise(flux_df_, metric="cosine")
flux_df_selected = flux_df_.loc[growth_df[growth_df["is_experimental_essential"]].index].copy()
pca_ = PCA(n_components=25)
flux_df_pca = pd.DataFrame(
    pca_.fit_transform(flux_df_selected),
    index=flux_df_selected.index,
)
flux_df_vis = construct_vis(flux_df_pca, n_clusters=8)

flux_pairwise_ = compute_pairwise(flux_df_pca, metric="euclidean")

In [ ]:
fig = (
    gg.ggplot(flux_df_vis, gg.aes(x="t-SNE1", y="t-SNE2", color="factor(cluster)"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(legend_key_size=8, figure_size=(4, 2))
    + gg.labs(title="t-SNE of MOMA fluxes")
)
fig.save("03122026_flux_tsne.png", dpi=300, bbox_inches="tight")
fig

### transcriptomics preprocessing

In [ ]:
selected_perts = z_transcript_case_df.index.isin(
    growth_df[growth_df["is_experimental_essential"]].index
)
transcript_df_selected = z_transcript_case_df.loc[selected_perts].copy()
transcript_df_selected = (
    transcript_df_selected - transcript_df_selected.mean(0)
) / transcript_df_selected.std(0)
transcript_df_vis = construct_vis(transcript_df_selected, n_clusters=8)

In [ ]:
fig = (
    gg.ggplot(transcript_df_vis, gg.aes(x="t-SNE1", y="t-SNE2", color="factor(cluster)"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(legend_key_size=8, figure_size=(4, 2))
    + gg.labs(title="t-SNE of transcriptomic data")
)
fig.save("03122026_transcript_tsne.png", dpi=300, bbox_inches="tight")
fig

In [ ]:
joint_vis = flux_df_vis.merge(
    transcript_df_vis,
    left_index=True,
    right_index=True,
    how="inner",
    suffixes=("_flux", "_transcript"),
)
joint_vis

In [ ]:
# fig = (
#     gg.ggplot(
#         joint_vis, gg.aes(x="t-SNE1_flux", y="t-SNE2_flux", color="factor(cluster_transcript)")
#     )
#     + gg.geom_point(size=0.5)
#     + gg.theme_minimal()
#     + SHARED_THEME
#     + gg.theme(legend_key_size=8, figure_size=(4, 2))
#     + gg.labs(title="t-SNE of MOMA fluxes")
# )
# fig.save("03122026_flux_tsne.png", dpi=300, bbox_inches="tight")
# fig

In [ ]:
fig = (
    gg.ggplot(
        joint_vis,
        gg.aes(x="t-SNE1_transcript", y="t-SNE2_transcript", color="factor(cluster_flux)"),
    )
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(legend_key_size=8, figure_size=(4, 2))
    + gg.labs(title="t-SNE of transcriptomic data", color="flux cluster")
)
fig.save("03122026_transcript_tsne_flux_cluster.png", dpi=300, bbox_inches="tight")
fig

In [ ]:
vis_subset = transcript_df_vis.loc[lambda x: x.index.str.startswith("lpx")].assign(
    label=lambda x: x.index
)

fig = (
    gg.ggplot(transcript_df_vis, gg.aes(x="t-SNE1", y="t-SNE2", color="factor(cluster)"))
    + gg.geom_point(size=0.5)
    + gg.geom_text(vis_subset, gg.aes(label="label"), color="black", size=5)
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(legend_key_size=8, figure_size=(4, 2), legend_position="none")
    + gg.labs(title="t-SNE of transcriptomic data")
)
fig.save("03122026_transcript_tsne_lpx.png", dpi=300, bbox_inches="tight")
fig

In [ ]:
vis_subset = flux_df_vis.loc[lambda x: x.index.str.startswith("lpx")].assign(
    label=lambda x: x.index
)

fig = (
    gg.ggplot(flux_df_vis, gg.aes(x="t-SNE1", y="t-SNE2", color="factor(cluster)"))
    + gg.geom_point(size=0.5)
    + gg.geom_text(vis_subset, gg.aes(label="label"), color="black", size=5)
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(legend_key_size=8, figure_size=(4, 2), legend_position="none")
    + gg.labs(title="t-SNE of flux data")
)
fig.save("03122026_flux_tsne_lpx.png", dpi=300, bbox_inches="tight")
fig

In [ ]:
joint_vis.groupby("cluster_transcript").apply(lambda x: print(x.name, ", ".join(x.index)))

In [ ]:
joint_vis.groupby("cluster_flux").apply(lambda x: print(x.name, ", ".join(x.index)))

In [ ]:
# Transcriptomics
transcript_pairwise_ = compute_pairwise(z_transcript_case_df, metric="euclidean")
# transcript_pairwise_case = compute_pairwise(z_transcript_case_df, metric="cosine")
predicted_essential_perts = growth_df.loc[growth_df["is_predicted_essential"]].index
perts_inter = np.intersect1d(predicted_essential_perts, transcript_pairwise_.index)


# Transcriptomics - all
# transcript_pairwise_ = compute_pairwise(z_transcript_df, metric="euclidean")
# perts_inter = np.intersect1d(
#     flux_pairwise_.index.astype(str), transcript_pairwise_.index.astype(str)
# )

### Plots

In [ ]:
transcript_pairwise_aligned = transcript_pairwise_.loc[perts_inter].loc[:, perts_inter]
flux_pairwise_aligned = flux_pairwise_.loc[perts_inter].loc[:, perts_inter]

row_idx, col_idx = np.triu_indices_from(transcript_pairwise_aligned.values, k=1)
plot_df = pd.DataFrame(
    {
        "pert1": transcript_pairwise_aligned.index[row_idx],
        "pert2": transcript_pairwise_aligned.columns[col_idx],
        "flux_similarity": flux_pairwise_aligned.values[row_idx, col_idx],
        "transcript_similarity": transcript_pairwise_aligned.values[row_idx, col_idx],
    }
)

plot_df["flux_rank"] = plot_df["flux_similarity"].rank()
plot_df["transcript_rank"] = plot_df["transcript_similarity"].rank()

fig = (
    gg.ggplot(plot_df, gg.aes(x="flux_rank", y="transcript_rank"))
    + gg.geom_bin2d(bins=20)
    + gg.theme_minimal()
)
display(fig)

fig2 = (
    gg.ggplot(plot_df, gg.aes(x="flux_rank", y="transcript_rank"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
)
display(fig2)

fig3 = gg.ggplot(plot_df, gg.aes(x="flux_similarity", y="transcript_similarity")) + gg.geom_point(
    size=0.5
)
display(fig3)

In [ ]:
stats.spearmanr(plot_df["flux_similarity"], plot_df["transcript_similarity"])

In [ ]:
sns.clustermap(flux_pairwise_pred_essential, xticklabels=False, yticklabels=True, figsize=(20, 20))

In [ ]:
sns.clustermap(
    transcript_pairwise_pred_essential, xticklabels=False, yticklabels=True, figsize=(20, 20)
)

In [ ]:
# # 1. Create the first clustermap and save its object
# cg1 = sns.clustermap(
#     flux_pairwise_pred_essential, xticklabels=False, yticklabels=True, figsize=(20, 20)
# )

# # 2. Pass its row and column linkages to the second clustermap
# cg2 = sns.clustermap(
#     transcript_pairwise_pred_essential,
#     row_linkage=cg1.dendrogram_row.linkage,
#     col_linkage=cg1.dendrogram_col.linkage,
#     xticklabels=False,
#     yticklabels=True,
#     figsize=(20, 20),
# )
# plt.show()

In [ ]:
# # 1. Create the first clustermap and save its object
# cg1 = sns.clustermap(
#     transcript_pairwise_pred_essential, xticklabels=False, yticklabels=True, figsize=(20, 20)
# )

# # 2. Pass its row and column linkages to the second clustermap
# cg2 = sns.clustermap(
#     flux_pairwise_pred_essential,
#     row_linkage=cg1.dendrogram_row.linkage,
#     col_linkage=cg1.dendrogram_col.linkage,
#     xticklabels=False,
#     yticklabels=True,
#     figsize=(20, 20),
# )
# plt.show()

In [ ]:
agreement_df = []
for gene in flux_pairwise_aligned.index:
    corr_ = stats.spearmanr(flux_pairwise_aligned.loc[gene], transcript_pairwise_aligned.loc[gene])
    agreement_df.append(
        {
            "gene": gene,
            "corr": corr_.correlation,
        }
    )
agreement_df = pd.DataFrame(agreement_df)

In [ ]:
agreement_df.hist(bins=10)
plt.show()

In [ ]:
g1 = agreement_df.sort_values("corr", ascending=False).tail(20)
print("genes wih poor agreement:")
print(", ".join(g1["gene"]))

g2 = agreement_df.sort_values("corr", ascending=True).tail(20)
print("genes with good agreement:")
print(", ".join(g2["gene"]))

# OLD

In [ ]:
rank_corr = stats.spearmanr(growth_df["growth_ratio"], growth_df["T4"])

fig = (
    gg.ggplot(growth_df, gg.aes(x="growth_ratio", y="T4"))
    + gg.geom_point(size=0.5)
    + gg.geom_smooth(method="lm")
    + gg.labs(
        x="FBA-predicted growth",
        y="experimental fitness",
        title=f"Spearman rho: {rank_corr.correlation:.2f}",
    )
    + gg.theme_minimal()
    + SHARED_THEME
)
display(fig)
# fig.save("growth_vs_fitness.png", dpi=300)

In [ ]:
def compute_jaccard(v1, v2, ks=None):
    """Jaccard index between the top-k elements of two vectors, for each k."""
    if ks is None:
        ks = [5, 10, 25, 50]
    v1 = np.asarray(v1)
    v2 = np.asarray(v2)
    n = min(len(v1), len(v2))
    order1 = np.argsort(v1)[::-1]
    order2 = np.argsort(v2)[::-1]
    results = []
    for k in ks:
        k_ = min(k, n)
        top1 = set(order1[:k_])
        top2 = set(order2[:k_])
        intersect_ = len(top1 & top2)
        union_ = len(top1 | top2)
        # print(intersect_, union_)
        results.append(intersect_ / union_)
    return np.array(results)


def compute_jaccard_scores(flux_pairwise, transcript_pairwise):
    jaccard_scores = []
    for gene_kd in tqdm(flux_pairwise.index):
        v1 = flux_pairwise.loc[gene_kd]
        v2 = transcript_pairwise.loc[gene_kd]

        v1_ = v1.loc[lambda x: x.index != gene_kd]
        v2_ = v2.loc[lambda x: x.index != gene_kd]
        jaccard_scores.append(compute_jaccard(v1_, v2_))
    jaccard_scores = (
        pd.DataFrame(
            jaccard_scores,
            index=flux_pairwise.index,
            columns=["k=5", "k=10", "k=25", "k=50"],
        )
        .stack()
        .to_frame("Jaccard")
        .reset_index()
        .rename(columns={"level_1": "k", "level_0": "gene"})
    )
    return jaccard_scores

In [ ]:
transcript_pairwise.index = transcript_pairwise.index.astype(str)
flux_pairwise.index = flux_pairwise.index.astype(str)
transcript_pairwise.columns = transcript_pairwise.columns.astype(str)
flux_pairwise.columns = flux_pairwise.columns.astype(str)
transcript_pairwise_case.index = transcript_pairwise_case.index.astype(str)
transcript_pairwise_case.columns = transcript_pairwise_case.columns.astype(str)

inter_intervals = np.intersect1d(transcript_pairwise.index, flux_pairwise.index)
transcript_pairwise_ = transcript_pairwise.loc[inter_intervals].loc[:, inter_intervals]
flux_pairwise_ = flux_pairwise.loc[inter_intervals].loc[:, inter_intervals]
jaccard_scores = compute_jaccard_scores(flux_pairwise_, transcript_pairwise_)

inter_intervals_case = np.intersect1d(transcript_pairwise_case.index, flux_pairwise.index)
transcript_pairwise_case_ = transcript_pairwise_case.loc[inter_intervals_case].loc[
    :, inter_intervals_case
]
flux_pairwise_2 = flux_pairwise.loc[inter_intervals_case].loc[:, inter_intervals_case]
jaccard_scores_case = compute_jaccard_scores(flux_pairwise_2, transcript_pairwise_case_)

In [ ]:
lpx_genes = ["lpxA", "lpxC", "lpxD", "lpxB", "lpxK"]
sns.heatmap(flux_pairwise.loc[lpx_genes, lpx_genes], cmap="Reds")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2))
flux_pairwise.loc["lpxA"].hist()
plt.xlabel("flux distance to lpxA flux (FBA)")
plt.ylabel("# of gene perturbations")
plt.show()

In [ ]:
growth_df

In [ ]:
growth_df.loc[lpx_genes]

In [ ]:
(
    gg.ggplot(jaccard_scores, gg.aes(y="Jaccard", x="factor(k)"))
    + gg.geom_boxplot()
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(jaccard_scores_case, gg.aes(y="Jaccard", x="factor(k)"))
    + gg.geom_boxplot()
    + gg.theme_minimal()
)

In [ ]:
dists_to_ctrl = pd.merge(
    transcript_d_to_ctrl,
    flux_d_to_ctrl,
    left_index=True,
    right_index=True,
    how="inner",
)
dists_to_ctrl

In [ ]:
corr_ = stats.spearmanr(dists_to_ctrl["flux_d_to_ctrl"], dists_to_ctrl["transcript_d_to_ctrl"])
fig = (
    gg.ggplot(dists_to_ctrl, gg.aes(x="flux_d_to_ctrl", y="transcript_d_to_ctrl"))
    + gg.geom_point()
    + gg.theme_minimal()
    + gg.labs(
        x="distance to control (flux)",
        y="distance to control (transcript)",
        title=f"Spearman rho: {corr_.correlation:.2f}",
    )
)
display(fig)

In [ ]:
flux_d_to_ctrl